In [9]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import json
import re
from deep_translator import GoogleTranslator

In [10]:
def to_json(data):
    """
    This function takes list of dictionary as Input and 
    then Creates a JSON file in which Input data is stored
    """
    with open("data_dict.json", "w") as outfile:
        json.dump(data, outfile,indent=4)
        outfile.close()

In [11]:
def get_data(slug_name):
    data_list = []
    url = "https://publicsite.dps.texas.gov/SexOffenderRegistry/map/load?mapReqId=1&channel=p-SexOffenderJs&address=Austin%2C+TX%2C+USA#"
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized") 
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--log-level=3")
    translator = GoogleTranslator(target='english')
    driver = webdriver.Chrome(options=options)
    driver.get(url)
    driver.find_element(By.XPATH, f'/html/body/form/div/div[2]/div[2]/div[2]/a').click()
    time.sleep(3)
    list1 = driver.find_elements(By.XPATH, f'/html/body/form/div/div[2]/div[3]/div[1]/div[2]/table/tbody/tr')
    # print(len(list1))
    for i in range (1, len(list1)+1):
        link = driver.find_element(By.XPATH, f'/html/body/form/div/div[2]/div[3]/div[1]/div[2]/table/tbody/tr[{i}]/td[2]/a').get_attribute("href")
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])
        driver.get(link)
        time.sleep(2)
        try:
            image = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div[2]/img').get_attribute('src')
            print(image)
        except:
            pass
        data_dict = {}
        crimeInfo = ""
        offense = ""
        additionalInfo = ""
        list2 = driver.find_elements(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div[3]/table/tbody/tr')
        fullName = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/h1').text
        firstName = fullName.split(",")[1].strip()
        lastName = fullName.split(",")[0].strip()
        fullName = firstName + " " + lastName
        print(fullName)
        for j in range(1, len(list2)+1):
            heading = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div[3]/table/tbody/tr[{j}]').text
            if "SID" in heading:
                sid = heading.split("SID")[1].strip()
                identifierType = "SID: " + sid
    #             print(identifierType)
            if "Risk Level" in heading:
                riskLevel = heading.split("Risk Level")[1].strip()
    #             print(riskLevel)
            if "Sex" in heading:
                gender = heading.split("Sex")[1].strip()
    #             print(gender)
            if "Race" in heading:
                race = heading.split("Race")[1].strip()
    #             print(race)
            if "Height" in heading:
                height = heading.split("Height")[1].strip()
    #             print(height)
            if "Weight" in heading:
                weight = heading.split("Weight")[1].strip()
    #             print(weight)
            if "Hair Color" in heading:
                hair = heading.split("Hair Color")[1].strip()
    #             print(hair)
            if "Eye Color" in heading:
                eyes = heading.split("Eye Color")[1].strip()
    #             print(eyes)
            if "Shoe Size" in heading:
                shoeSize = heading.split("Shoe Size")[1].strip() 
                additionalInfo = additionalInfo + "Shoe Size: " + shoeSize
            if "Shoe Width" in heading:
                shoeWidth = heading.split("Shoe Width")[1].strip()
                additionalInfo = additionalInfo + "; Shoe Width: "+  shoeWidth
        alias = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/ul[1]').text
        alias = alias.replace("\n", "; ")
        listOfDob = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/ul[2]').text.replace("\n", "; ")
        dob = listOfDob.split("(")[0].strip()
        fullAddress = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div[5]/table/tbody/tr/td').text.replace("\n", ", ")
    #     print(alias)
    #     print(listOfDob)
        list3 = driver.find_elements(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/h2')
        for k in range(1, len(list3)+1):
            head = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/h2[{k}]').text
            if "Offense" in head:
                if head.split("Offense:")[1].strip() not in offense:
                    offense = head.split("Offense:")[1].strip() + "; " + offense
        print(offense)
        list4 = driver.find_elements(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div/table')
        print(len(list4))
        for l in range(7, 2*len(list4)):
            try:
                crime = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div[{l}]/table').text.replace("\n", ", ")
    #             print(crime)
                crimeInfo = crime + "; " + crimeInfo
            except:
                pass
        print(crimeInfo)
        summary = fullName + " is one of the unmapped sex offenders."
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        print("*"*50)
        if fullName:
            data_dict['fullName'] = fullName
        if alias:
            data_dict['alias'] = alias
        if image:
            data_dict['image'] = image
        if offense:
            data_dict['offence'] = offense
        if dob:
            data_dict['dob'] = dob
        if listOfDob:
            data_dict['listOfDob'] = listOfDob
        if riskLevel:
            data_dict['riskLevel'] = riskLevel
        if gender:
            data_dict['gender'] = gender
        if race:
            data_dict['race'] = race
        if height:
            data_dict['height'] = height
        if weight:
            data_dict['weight'] = weight
        if hair:
            data_dict['hair'] = hair
        if eyes:
            data_dict['eyes'] = eyes
        if fullAddress:
            data_dict['fullAddress'] = fullAddress
        if identifierType:
            data_dict['identifierType'] = identifierType
        if crimeInfo:
            data_dict['crimeInfo'] = crimeInfo
        if additionalInfo:
            data_dict['additionalInfo'] = additionalInfo
        if summary:
            data_dict['summary'] = summary
        data_list.append(data_dict)
    i=1
    driver.find_element(By.XPATH, f'/html/body/form/div/div[2]/div[2]/div[3]/a').click()
    list1 = driver.find_element(By.XPATH, f'/html/body/form/div/div[2]/div[3]/div[2]/div[2]/table/tbody/tr')
    for i in range(1, len(list1)+1):
        link = driver.find_element(By.XPATH, f'/html/body/form/div/div[2]/div[3]/div[2]/div[2]/table/tbody/tr[{i}]/td[2]/a').get_attribute("href")
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])
        driver.get(link)
        time.sleep(2)
        image = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div[2]/img').get_attribute('src')
        print(image)
        data_dict2 = {}
        crimeInfo = ""
        offense = ""
        additionalInfo = ""
        list2 = driver.find_elements(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div[3]/table/tbody/tr')
        fullName = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/h1').text
        firstName = fullName.split(",")[1].strip()
        lastName = fullName.split(",")[0].strip()
        fullName = firstName + " " + lastName
        print(fullName)
        for j in range(1, len(list2)+1):
            heading = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div[3]/table/tbody/tr[{j}]').text
            if "SID" in heading:
                sid = heading.split("SID")[1].strip()
                identifierType = "SID: " + sid
    #             print(identifierType)
            if "Risk Level" in heading:
                riskLevel = heading.split("Risk Level")[1].strip()
    #             print(riskLevel)
            if "Sex" in heading:
                gender = heading.split("Sex")[1].strip()
    #             print(gender)
            if "Race" in heading:
                race = heading.split("Race")[1].strip()
    #             print(race)
            if "Height" in heading:
                height = heading.split("Height")[1].strip()
    #             print(height)
            if "Weight" in heading:
                weight = heading.split("Weight")[1].strip()
    #             print(weight)
            if "Hair Color" in heading:
                hair = heading.split("Hair Color")[1].strip()
    #             print(hair)
            if "Eye Color" in heading:
                eyes = heading.split("Eye Color")[1].strip()
    #             print(eyes)
            if "Shoe Size" in heading:
                shoeSize = heading.split("Shoe Size")[1].strip() 
                additionalInfo = additionalInfo + "Shoe Size: " + shoeSize
            if "Shoe Width" in heading:
                shoeWidth = heading.split("Shoe Width")[1].strip()
                additionalInfo = additionalInfo + "; Shoe Width: "+  shoeWidth
        alias = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/ul[1]').text
        alias = alias.replace("\n", "; ")
        listOfDob = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/ul[2]').text.replace("\n", "; ")
        dob = listOfDob.split("(")[0].strip()
        fullAddress = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div[5]/table/tbody/tr/td').text.replace("\n", ", ")
    #     print(alias)
    #     print(listOfDob)
        list3 = driver.find_elements(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/h2')
        for k in range(1, len(list3)+1):
            head = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/h2[{k}]').text
            if "Offense" in head:
                if head.split("Offense:")[1].strip() not in offense:
                    offense = head.split("Offense:")[1].strip() + "; " + offense
        print(offense)
        list4 = driver.find_elements(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div/table')
        print(len(list4))
        for l in range(7, 2*len(list4)):
            try:
                crime = driver.find_element(By.XPATH, f'/html/body/div[2]/div[2]/main/div[4]/div/div[{l}]/table').text.replace("\n", ", ")
    #             print(crime)
                crimeInfo = crime + "; " + crimeInfo
            except:
                pass
        print(crimeInfo)
        summary = fullName + " is one of the mapped sex offenders."
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        print("*"*50)
        if fullName:
            data_dict2['fullName'] = fullName
        if alias:
            data_dict2['alias'] = alias
        if image:
            data_dict2['image'] = image
        if offense:
            data_dict2['offence'] = offense
        if dob:
            data_dict2['dob'] = dob
        if listOfDob:
            data_dict2['listOfDob'] = listOfDob
        if riskLevel:
            data_dict2['riskLevel'] = riskLevel
        if gender:
            data_dict2['gender'] = gender
        if race:
            data_dict2['race'] = race
        if height:
            data_dict2['height'] = height
        if weight:
            data_dict2['weight'] = weight
        if hair:
            data_dict2['hair'] = hair
        if eyes:
            data_dict2['eyes'] = eyes
        if fullAddress:
            data_dict2['fullAddress'] = fullAddress
        if identifierType:
            data_dict2['identifierType'] = identifierType
        if crimeInfo:
            data_dict2['crimeInfo'] = crimeInfo
        if additionalInfo:
            data_dict2['additionalInfo'] = additionalInfo
        if summary:
            data_dict2['summary'] = summary
        data_list.append(data_dict)
    driver.quit()
    return data_list


In [12]:
if __name__ == '__main__':
    data_list = get_data("add_slug_name")
    to_json(data_list)

https://publicsite.dps.texas.gov/SexOffenderRegistry/Search/Rapsheet/CurrentPhoto?Sid=08702469
KEVIN DAVEEN COMEAUX
SEXUAL ASSAULT; 
5
Statute TEXAS PENAL CODE 22.011, Victim Sex Female, Victim Age 18, Disposition Date 10/09/2015, JUDGMENT 5Y DISCHARGED FROM INCARCERATION; 
**************************************************
https://publicsite.dps.texas.gov/SexOffenderRegistry/Search/Rapsheet/CurrentPhoto?Sid=50035438
SAMUEL SANTOS
INDECENCY WITH A CHILD BY EXPOSURE; 
6
Statute TEXAS PENAL CODE 21.11(a)(2), Victim Sex Female, Victim Age 13, Disposition Date 11/29/2018, JUDGMENT 3Y DISCHARGED FROM INCARCERATION; Statute TEXAS PENAL CODE 21.11(a)(2), Victim Sex Female, Victim Age 13, Disposition Date 11/29/2018, JUDGMENT 3Y DISCHARGED FROM INCARCERATION; 
**************************************************
https://publicsite.dps.texas.gov/SexOffenderRegistry/Search/Rapsheet/CurrentPhoto?Sid=07558079
EDGAR DIAZ AGUADO
INDECENCY WITH A CHILD BY EXPOSURE; 
5
Statute TEXAS PENAL CODE 21.11(a)(

https://publicsite.dps.texas.gov/SexOffenderRegistry/Search/Rapsheet/CurrentPhoto?Sid=06577935
JOSE S CARRASCO
INDECENCY WITH A CHILD BY CONTACT; AGGRAVATED SEXUAL ASSAULT OF A CHILD; 
6
Statute TEXAS PENAL CODE 21.11(a)(1), Victim Sex Female, Victim Age 11, Disposition Date 10/28/2005, JUDGMENT 12Y DISCHARGED FROM INCARCERATION; Statute TEXAS PENAL CODE 22.021(a)(1)(B), Victim Sex Female, Victim Age 11, Disposition Date 10/28/2005, JUDGMENT 12Y DISCHARGED FROM INCARCERATION; 
**************************************************
https://publicsite.dps.texas.gov/SexOffenderRegistry/Search/Rapsheet/CurrentPhoto?Sid=05032799
EZELL JOHNSON
ATTEMPT TO COMMIT AGGRAVATED SEXUAL ASSAULT; AGGRAVATED KIDNAPPING WITH INTENT TO VIOLATE OR ABUSE THE VICTIM SEXUALLY; 
6
Statute TEXAS PENAL CODE 22.021 AND 15.01, Victim Sex Female, Victim Age 13, Disposition Date 07/21/1997, JUDGMENT 20Y DISCHARGED FROM INCARCERATION; Statute TEXAS PENAL CODE 20.04(a)(4), Victim Sex Female, Victim Age 13, Disposition D

https://publicsite.dps.texas.gov/SexOffenderRegistry/Search/Rapsheet/CurrentPhoto?Sid=16056505
FREDERICK CARATHER
COURT ORDERED TO REGISTER AS A CONDITION OF SUPERVISION FOR VIOLATION OF "INJURY TO A CHILD, ELDERLY INDIVIDUAL, OR DISABLED INDIVIDUAL WITH INTENT TO CAUSE SERIOUS BODILY INJURY"; 
5
Statute TEXAS PENAL CODE 22.04(e), Victim Sex Female, Victim Age 15, Disposition Date 05/08/2021, JUDGMENT 7YPROBATION/COMMUNITY SUPERVISION; 
**************************************************
https://publicsite.dps.texas.gov/SexOffenderRegistry/Search/Rapsheet/CurrentPhoto?Sid=16399391
JOHNNY LEE HARRIS
FORCIBLE SODOMY/FORCE; 
5
Statute OKLAHOMA STATUTE TITLE 21 SECTION 888 (A), Victim Sex Female, Victim Age 18, Disposition Date 05/10/2006, JUDGMENT 10YPROBATION/COMMUNITY SUPERVISION; 
**************************************************


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"/html/body/div[2]/div[2]/main/div[4]/div/div[2]/img"}
  (Session info: chrome=104.0.5112.81)
Stacktrace:
Backtrace:
	Ordinal0 [0x006D78B3+2193587]
	Ordinal0 [0x00670681+1771137]
	Ordinal0 [0x005841A8+803240]
	Ordinal0 [0x005B24A0+992416]
	Ordinal0 [0x005B273B+993083]
	Ordinal0 [0x005DF7C2+1177538]
	Ordinal0 [0x005CD7F4+1103860]
	Ordinal0 [0x005DDAE2+1170146]
	Ordinal0 [0x005CD5C6+1103302]
	Ordinal0 [0x005A77E0+948192]
	Ordinal0 [0x005A86E6+952038]
	GetHandleVerifier [0x00980CB2+2738370]
	GetHandleVerifier [0x009721B8+2678216]
	GetHandleVerifier [0x007617AA+512954]
	GetHandleVerifier [0x00760856+509030]
	Ordinal0 [0x0067743B+1799227]
	Ordinal0 [0x0067BB68+1817448]
	Ordinal0 [0x0067BC55+1817685]
	Ordinal0 [0x00685230+1856048]
	BaseThreadInitThunk [0x7548FA29+25]
	RtlGetAppContainerNamedObjectPath [0x76FD7A9E+286]
	RtlGetAppContainerNamedObjectPath [0x76FD7A6E+238]
